# 第1部分：导入库和初始化

In [1]:
import transformers
print(transformers.__version__)

4.53.3


In [2]:
# 确定编码工具
from transformers import BertTokenizer
# 读取数据集  
import pandas as pd
from datasets import Dataset
# 定义数据集
import torch
from torch.optim import AdamW
from transformers.optimization import get_scheduler
from transformers import BertModel
import torch.nn as nn

# 确定编码工具
token = BertTokenizer.from_pretrained('bert-base-chinese')
print(token)

device = 'cuda'  # 此处容易报错RuntimeError: Expected one of cpu, cuda, xpu, mkldnn, opengl, opencl,
                 #   ideep, hip, ve, ort, mlc, xla, lazy, vulkan, meta, hpu device type at
                 #   start of device string: gpu
                 #   ———————————————
if torch.cuda.is_available():
    device = 'cuda'
print(device)

2025-12-06 05:28:56.126816: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764998936.294352      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764998936.344673      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

BertTokenizer(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
cuda


# 第2部分：加载数据

In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
   for filename in filenames:
       print(os.path.join(dirname, filename))

/kaggle/input/train_split.csv
/kaggle/input/test_final.csv
/kaggle/input/validation.csv


In [4]:
import pandas as pd
# 读取已保存的 CSV 文件
train_data = pd.read_csv('/kaggle/input/train_split.csv')
test_data = pd.read_csv('/kaggle/input/test_final.csv')
val_data = pd.read_csv('/kaggle/input/validation.csv')

# 构建 Hugging Face 数据集
train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)
val_dataset = Dataset.from_pandas(val_data)

print(train_dataset)
print(test_dataset)
print(val_dataset)
print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

Dataset({
    features: ['cat', 'label', 'review'],
    num_rows: 39636
})
Dataset({
    features: ['cat', 'label', 'review'],
    num_rows: 12378
})
Dataset({
    features: ['cat', 'label', 'review'],
    num_rows: 9910
})
训练集大小: 39636
验证集大小: 9910
测试集大小: 12378


# 第3部分：定义自定义数据集类

In [5]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, item):
        text = self.dataset[item]['review']
        label = self.dataset[item]['label']
        return text, label

# 使用示例
train_dataset = CustomDataset(train_dataset)
test_dataset = CustomDataset(test_dataset)
val_dataset = CustomDataset(val_dataset)

print(f"训练集样本数: {len(train_dataset)}")
print(f"示例数据: {train_dataset[20]}")

训练集样本数: 39636
示例数据: ('从来没在京东上给过差评，买回来的就像刚刚灌在瓶子里就拿出来的一样！', 0)


# 第4部分：定义数据整理函数和加载器

In [6]:
# 定义数据整理函数
def collate_fn(data):
    sents = [i[0] for i in data]
    labels = [i[1] for i in data]
    data = token.batch_encode_plus(batch_text_or_text_pairs=sents,
                                   truncation=True,
                                   padding='max_length',
                                   max_length=500,
                                   return_tensors='pt',
                                   return_length=True)
    input_ids = data['input_ids']
    attention_mask = data['attention_mask']
    token_type_ids = data['token_type_ids']
    labels = torch.tensor([label if label != "-1" else "0" for label in labels]).long() # 这个地方导致三分类变为二分类
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    token_type_ids = token_type_ids.to(device)
    labels = labels.to(device)
    
    return input_ids, attention_mask, token_type_ids, labels

# 定义数据集加载器
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                     batch_size=16,
                                     collate_fn=collate_fn,
                                     shuffle=True,
                                     drop_last=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                     batch_size=16,
                                     collate_fn=collate_fn,
                                     shuffle=True,
                                     drop_last=True)
val_loader = torch.utils.data.DataLoader(dataset=val_dataset,
                                     batch_size=16,
                                     collate_fn=collate_fn,
                                     shuffle=True,
                                     drop_last=True)

print(f"训练集batch数: {len(train_loader)}")
print(f"验证集batch数: {len(val_loader)}")
print(f"测试集batch数: {len(test_loader)}")

训练集batch数: 2477
验证集batch数: 619
测试集batch数: 773


# 第5部分：加载预训练BERT模型

In [7]:
# 加载预训练BERT模型
pretrained = BertModel.from_pretrained('bert-base-chinese').to(device)
for param in pretrained.parameters():
    param.requires_grad_(False)
print("BERT模型加载完成，参数已冻结")

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

BERT模型加载完成，参数已冻结


# 第6部分：定义BertBiLSTM模型

In [8]:
class BertBiLSTMClassifier(nn.Module):
    def __init__(self, num_classes, hidden_size=768, lstm_hidden_size=128, lstm_layers=1):
        super(BertBiLSTMClassifier, self).__init__()
        # BiLSTM层
        self.lstm = nn.LSTM(input_size=hidden_size, hidden_size=lstm_hidden_size, num_layers=lstm_layers,
                            batch_first=True, bidirectional=True)
        # 全连接层用于分类
        self.fc = nn.Linear(lstm_hidden_size * 2, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids):
        # BERT的前向传播
        with torch.no_grad():
            outputs = pretrained(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        last_hidden_state = outputs.last_hidden_state
        # 将BERT输出输入BiLSTM
        lstm_out, _ = self.lstm(last_hidden_state)
        # 提取BiLSTM的最后一层输出
        lstm_out = lstm_out[:, -1, :]
        # 全连接层分类
        logits = self.fc(lstm_out)
        return logits

# 定义模型和输入
num_classes = 3  # 你的分类类别数量
model = BertBiLSTMClassifier(num_classes).to(device)

# 输出模型结构
print("模型结构:")
print(model)

模型结构:
BertBiLSTMClassifier(
  (lstm): LSTM(768, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)


# 第7部分：定义优化器和损失函数

In [9]:
# 定义优化器、损失函数和学习率调度器
optimizer = AdamW(model.parameters(), lr=5e-4)
criterion = torch.nn.CrossEntropyLoss()
scheduler = get_scheduler(name='linear', num_warmup_steps=0, num_training_steps=len(train_loader), optimizer=optimizer)

print("优化器、损失函数和调度器定义完成")
print(f"学习率: {optimizer.param_groups[0]['lr']}")
print(f"损失函数: {criterion}")

优化器、损失函数和调度器定义完成
学习率: 0.0005
损失函数: CrossEntropyLoss()


# 第8部分：训练和验证函数

In [10]:
def train_and_validate(model, train_loader, val_loader, optimizer, criterion, scheduler, num_epochs=2):
    for epoch in range(num_epochs):
        model.train()
        for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(train_loader):
            out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if i % 10 == 0:
                out = out.argmax(dim=1)
                accuracy = (out == labels).sum().item() / len(labels)
                lr = optimizer.state_dict()['param_groups'][0]['lr']
                print(f'Train Epoch {epoch}, Step {i}, Loss: {loss.item():.4f}, LR: {lr:.6f}, Accuracy: {accuracy:.4f}')

            if i % 100 == 0:
                torch.save(model.state_dict(), f'bert_bilstm_model_epoch_{epoch}_step_{i}.pth')

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for val_input_ids, val_attention_mask, val_token_type_ids, val_labels in val_loader:
                val_out = model(input_ids=val_input_ids, attention_mask=val_attention_mask, token_type_ids=val_token_type_ids)
                val_loss += criterion(val_out, val_labels).item()
                val_out = val_out.argmax(dim=1)
                correct += (val_out == val_labels).sum().item()
                total += len(val_labels)

        val_accuracy = correct / total
        average_val_loss = val_loss / len(val_loader)
        print(f'Validation Epoch {epoch}, Loss: {average_val_loss:.4f}, Accuracy: {val_accuracy:.4f}')
        print('-' * 50)

print("训练和验证函数定义完成")

训练和验证函数定义完成


# 第9部分：训练模型

In [11]:
# 开始训练
print("开始训练模型...")
train_and_validate(model, train_loader, val_loader, optimizer, criterion, scheduler, num_epochs=2)
print("训练完成!")

开始训练模型...
Train Epoch 0, Step 0, Loss: 1.0384, LR: 0.000500, Accuracy: 0.5000
Train Epoch 0, Step 10, Loss: 0.6416, LR: 0.000498, Accuracy: 0.9375
Train Epoch 0, Step 20, Loss: 0.6014, LR: 0.000496, Accuracy: 0.7500
Train Epoch 0, Step 30, Loss: 0.4696, LR: 0.000494, Accuracy: 0.7500
Train Epoch 0, Step 40, Loss: 0.3105, LR: 0.000492, Accuracy: 0.8750
Train Epoch 0, Step 50, Loss: 0.3603, LR: 0.000490, Accuracy: 0.8125
Train Epoch 0, Step 60, Loss: 0.3399, LR: 0.000488, Accuracy: 0.8750
Train Epoch 0, Step 70, Loss: 0.4397, LR: 0.000486, Accuracy: 0.8125
Train Epoch 0, Step 80, Loss: 0.3637, LR: 0.000484, Accuracy: 0.8750
Train Epoch 0, Step 90, Loss: 0.2060, LR: 0.000482, Accuracy: 0.9375
Train Epoch 0, Step 100, Loss: 0.3057, LR: 0.000480, Accuracy: 0.8125
Train Epoch 0, Step 110, Loss: 0.7058, LR: 0.000478, Accuracy: 0.6875
Train Epoch 0, Step 120, Loss: 0.5547, LR: 0.000476, Accuracy: 0.7500
Train Epoch 0, Step 130, Loss: 0.2818, LR: 0.000474, Accuracy: 0.8750
Train Epoch 0, Step 1

# 第10部分：测试和评估模型

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def test():
    model.eval()
    correct = 0
    total = 0
    y_true = []
    y_pred = []

    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(test_loader):
        if i == 200:  # 限制测试batch数，可以调整或删除
            break

        with torch.no_grad():
            out = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
            out = torch.argmax(out, dim=1)
            correct += (out == labels).sum().item()
            total += len(labels)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(out.cpu().numpy())
        print(f"Batch {i}: Accuracy: {correct/total:.4f}")

    # 计算评估指标
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    print("\n" + "="*50)
    print("模型测试结果:")
    print(f"Final Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("="*50)

print("开始测试模型...")
test()

开始测试模型...
Batch 0: Accuracy: 0.9375
Batch 1: Accuracy: 0.9688
Batch 2: Accuracy: 0.9375
Batch 3: Accuracy: 0.9531
Batch 4: Accuracy: 0.9375
Batch 5: Accuracy: 0.9479
Batch 6: Accuracy: 0.9286
Batch 7: Accuracy: 0.9219
Batch 8: Accuracy: 0.9167
Batch 9: Accuracy: 0.9250
Batch 10: Accuracy: 0.9318
Batch 11: Accuracy: 0.9323
Batch 12: Accuracy: 0.9327
Batch 13: Accuracy: 0.9375
Batch 14: Accuracy: 0.9375
Batch 15: Accuracy: 0.9336
Batch 16: Accuracy: 0.9338
Batch 17: Accuracy: 0.9340
Batch 18: Accuracy: 0.9342
Batch 19: Accuracy: 0.9313
Batch 20: Accuracy: 0.9315
Batch 21: Accuracy: 0.9347
Batch 22: Accuracy: 0.9321
Batch 23: Accuracy: 0.9323
Batch 24: Accuracy: 0.9275
Batch 25: Accuracy: 0.9279
Batch 26: Accuracy: 0.9306
Batch 27: Accuracy: 0.9263
Batch 28: Accuracy: 0.9224
Batch 29: Accuracy: 0.9208
Batch 30: Accuracy: 0.9214
Batch 31: Accuracy: 0.9238
Batch 32: Accuracy: 0.9223
Batch 33: Accuracy: 0.9228
Batch 34: Accuracy: 0.9232
Batch 35: Accuracy: 0.9236
Batch 36: Accuracy: 0.9223
B